# Green SM downstream evaluation

CPU-only evaluator for completed Food and Ads artifacts. Attach the linked full corpus plus one or more completed prediction-artifact datasets before running. The committed CLI owns all metric logic.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO = Path('/kaggle/working/repo')
if REPO.exists():
    raise RuntimeError('Fresh Kaggle session required: /kaggle/working/repo already exists.')
subprocess.run(['git', 'clone', '--branch', 'feat/foundation-encoder', 'https://github.com/huongiang171/green-sm-foundation-pretraining.git', str(REPO)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], cwd=REPO, check=True)
print(subprocess.check_output(['git', 'log', '-1', '--oneline'], cwd=REPO, text=True).strip())


In [ ]:
INPUT_ROOT = Path('/kaggle/input')
OUTPUT_ROOT = Path('/kaggle/working/downstream_evaluation')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def find_unique(name, required_relative):
    matches = [path for path in INPUT_ROOT.rglob(name) if path.is_dir() and (path / required_relative).is_file()]
    assert len(matches) == 1, f'Expected exactly one {name!r}; found {matches}'
    return matches[0]

CORPUS_ROOT = find_unique('green_sm_sim_full_v1', 'green_sm_sim_linked_v1/foundation/manifest.json')
LINKED_ROOT = CORPUS_ROOT / 'green_sm_sim_linked_v1'
OPPORTUNITY_ROOTS = {'food_recommendation': LINKED_ROOT / 'food_opportunities', 'ad_optimization': LINKED_ROOT / 'ads_opportunities'}
for task, root in OPPORTUNITY_ROOTS.items():
    assert (root / 'manifest.json').is_file(), f'Missing {task} opportunity manifest: {root}'
print('Linked corpus:', LINKED_ROOT)


In [ ]:
import json

def artifact_task(directory):
    baseline_sidecar = directory / 'evaluation_input_v1.json'
    foundation_manifest = directory / 'run_manifest.test.json'
    metadata_path = baseline_sidecar if baseline_sidecar.is_file() else foundation_manifest
    if not metadata_path.is_file():
        return None
    payload = json.loads(metadata_path.read_text(encoding='utf-8'))
    declared_task = payload.get('task')
    return declared_task if isinstance(declared_task, str) else None

def prediction_directories(task):
    candidates = []
    for prediction_file in INPUT_ROOT.rglob('predictions.test.parquet'):
        directory = prediction_file.parent
        has_foundation = (directory / 'run_manifest.test.json').is_file() and (directory / 'validation_report.test.json').is_file()
        has_baseline = (directory / 'evaluation_input_v1.json').is_file()
        if (has_foundation or has_baseline) and artifact_task(directory) == task:
            candidates.append(directory)
    selected = sorted(set(candidates))
    assert selected, f'Attach at least one completed {task} prediction artifact dataset.'
    return selected

REPORTS = {}
for task, opportunities_root in OPPORTUNITY_ROOTS.items():
    task_reports = []
    for index, prediction_dir in enumerate(prediction_directories(task), start=1):
        output = OUTPUT_ROOT / task / f'report_{index}.json'
        command = [sys.executable, '-m', 'src.evaluation', 'evaluate', '--task', task, '--split', 'test', '--prediction-dir', str(prediction_dir), '--opportunities-root', str(opportunities_root), '--output', str(output)]
        subprocess.run(command, cwd=REPO, check=True)
        task_reports.append(output)
    REPORTS[task] = task_reports
print({task: [str(path) for path in paths] for task, paths in REPORTS.items()})


In [ ]:
for task, report_paths in REPORTS.items():
    if len(report_paths) < 2:
        print(f'{task}: one report attached; comparison needs a baseline and Foundation report.')
        continue
    output = OUTPUT_ROOT / task / 'comparison.json'
    command = [sys.executable, '-m', 'src.evaluation', 'compare']
    for report_path in report_paths:
        command.extend(['--report', str(report_path)])
    command.extend(['--output', str(output)])
    subprocess.run(command, cwd=REPO, check=True)
    print(f'{task}: comparison written to {output}')

for path in sorted(OUTPUT_ROOT.rglob('*.json')):
    print(path, path.stat().st_size, 'bytes')
